# Datamine PPQQPLOT Process Example

This notebook demonstrates how to configure and run the **`ppqqplot`** process wrapper in `dmstudio`.

## Process Description

## PPQQPLOT

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process calculates and displays PP (probability-probability) and QQ (quantile-quantile) plots. These are graphical techniques for determining if two data sets come from populations with a common distribution.

#### PP Plots

The PP plot is a plot of the cumulative probability of the first data set against the cumulative probability of the second data set for a series of cutoff grades. If the two sets come from a population with the same distribution, the points should fall approximately along the 45 degree reference line.

In order to calculate the cumulative probability for each file, samples are divided into bins as defined by parameter BINSIZE.

#### QQ Plots

The QQ plot is a plot of the quantiles of the first data set against the quantiles of the second data set, where a quantile is defined as the fraction (or percent) of points below the given value. For example the 0.2 (or 20%) quantile is the point at which 20% percent of the data fall below and 80% fall above that value. If the two sets come from a population with the same distribution, the points should fall approximately along the 45 degree reference line.

**Note** : The q-q plot is similar to a probability plot. For a probability plot, the quantiles for one of the data samples are replaced with the quantiles of a theoretical distribution. Further information on QQ plots is available from the Engineering Statistics Handbook (<http://www.itl.nist.gov/div898/handbook/eda/section3/qqplot.htm>).

If there are 100 or more samples in each of the IN1 and IN2 files then the 1%, 2%, 3%, ..., 100% percentiles are calculated. If there are less than 100 samples in either file then quantiles are calculated in multiples of 100/N where N is number of samples in the smaller file.

#### Key Field

If a **KEY** field is specified then the field must exist in the **IN1** file. The same **KEY** field may also exist in the **IN2** file. A PP or QQ line or set of symbols will be created for each unique value of the **KEY** field.

If the **KEY** field only exists in the **IN1** file then all the samples from the **IN2** file will be matched with the samples from the **IN1** file for every **KEY** field value. This is particularly useful if you are comparing multiple realisations from a conditional simulation run with the original sample data as shown in example 3 below.

### Input Files:

* **in1** (Table):
  First input data file. There must be a minimum of 5 records.
  Required=Yes

* **in2** (Table):
  Second input data file. This can be the same as the first input file IN1. There must be
  a minimum of 5 records.
  Required=Yes

### Output Files:

* **ppout** (No):
  Table
  Required=No

* **ppplot** (No):
  Plot
  Required=No

* **qqout** (No):
  Table
  Required=No

* **qqplot** (No):
  Plot
  Required=No

### Fields:

* **values** (Undefined : Undefined):
  Field in input file **IN1** to be plotted along the X axis of the PP and QQ plot.
  Default=Undefined
  Required=No

* **weights** (Undefined : Undefined):
  Weighting field for **VALUE1** in input file **IN1**.
  Default=Undefined
  Required=No

* **key** (Numeric : IN1):
  Key field in the input **IN1** and optionally **IN2** file. The plot will include a
  line or set of symbols for each key field value.
  Default=Undefined
  Required=No

### Parameters:

* **minimum1**:
  Minimum value of **VALUE1** field in input file IN1. Values below the minimum are
  ignored.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **minimum2**:
  Minimum value of **VALUE2** field in input file IN2. Values below the minimum are
  ignored.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **maximum1**:
  Maximum value of **VALUE1** field in input file IN1. Values above the maximum are
  ignored.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maximum2**:
  Maximum value of **VALUE2** field in input file IN2. Values above the maximum are
  ignored.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **plottype**:
  Flag to specify plot type (1); =1 : Scatter plot, using symbol X; =2 : Line plot
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **binsize**:
  Bin size for PP plot.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **diagonal**:
  Flag to control whether the diagonal (45 degree line) should be included on the plot
  (1). =0 : no diagonal line; =1 : include diagonal line.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **progress**:
  Flag to control amount of output written to Output Window (1). =0 : no output; =1 :
  progress messages
  Range=0,1
  Values=0,1
  Default=1
  Required=No

* **display**:
  Flag to select whether or not to display plot files. =0 : do not display plot files. =1
  : display plot files.
  Range=0,1
  Values=0,1
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('ppqqplot')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute ppqqplot
print("Running ppqqplot...")
dm_cmd.ppqqplot(
    inmods_i=['optional'],  # required
    # ppout_o='t_ppqqplot_out',  # optional
    # ppplot_o='t_ppqqplot_out',  # optional
    # qqout_o='t_ppqqplot_out',  # optional
    # qqplot_o='t_ppqqplot_out',  # optional
    # values_f=['optional'],  # optional
    # weights_f=['optional'],  # optional
    # key_f=['BHID'],  # optional
    # minimums_f=['optional'],  # optional
    # maximums_f=['optional'],  # optional
    # plottype_p=1,  # optional
    # binsize_p=1,  # optional
    # diagonal_p=1,  # optional
    # progress_p=1,  # optional
    # display_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("ppqqplot execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_ppqqplot_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")